<a href="https://colab.research.google.com/github/Venkatyellala77/document-qa-agent-rag/blob/main/Doc_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anthropic chromadb sentence-transformers pypdf gradio


In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
import anthropic, chromadb, gradio
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
print("All imports OK")

In [6]:
import os
import gradio as gr
import chromadb
import anthropic
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL = "claude-haiku-5-5"  # replace with a current model name from your account
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma = chromadb.Client()
collection = None


def ask_llm(prompt):
    r = client.messages.create(
        model=MODEL, max_tokens=1500,
        messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in r.content if b.type == "text")


def chunk_text(text, size=1200, overlap=150):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i + size])
        i += size - overlap
    return out


def ingest(file):
    global collection
    if file is None:
        return "Upload a PDF first."
    try:
        chroma.delete_collection("docs")
    except Exception:
        pass
    collection = chroma.create_collection("docs")
    reader = PdfReader(file)
    ids, texts, metas = [], [], []
    for p, page in enumerate(reader.pages, 1):
        for j, c in enumerate(chunk_text(page.extract_text() or "")):
            ids.append(f"{p}-{j}")
            texts.append(c)
            metas.append({"page": p})
    if not texts:
        return "No text found. The PDF may be scanned."
    collection.add(ids=ids, documents=texts, metadatas=metas,
                   embeddings=embedder.encode(texts).tolist())
    return f"Indexed {len(texts)} chunks from {len(reader.pages)} pages."


def retrieve(query, k=8):
    res = collection.query(
        query_embeddings=embedder.encode([query]).tolist(), n_results=k)
    docs, metas = res["documents"][0], res["metadatas"][0]
    ctx = "\n\n".join(f"[Page {m['page']}] {d}" for d, m in zip(docs, metas))
    return ctx, list(zip(docs, metas))


def is_relevant(q, ctx):
    r = ask_llm(f"Question: {q}\n\nContext:\n{ctx}\n\n"
                "Does the context contain the answer? Reply only YES or NO.")
    return r.strip().upper().startswith("YES")


def rewrite(q):
    return ask_llm("Rewrite this as a better search query. "
                   f"Return only the query:\n{q}")


def ask(question):
    if collection is None:
        return "Please upload and index a PDF first.", ""
    query = question
    for attempt in range(3):  # first try plus 2 rewrites
        ctx, sources = retrieve(query)
        if is_relevant(question, ctx):
            reply = ask_llm(
                "Answer ONLY from the context. Cite pages like [Page X]. "
                "If the answer is not there, say you don't know.\n\n"
                f"Context:\n{ctx}\n\nQuestion: {question}")
            src = "\n\n".join(f"**Page {m['page']}:** {d[:300]}..."
                              for d, m in sources)
            return reply, src
        query = rewrite(query)
    return "I don't know based on the provided document.", ""


with gr.Blocks() as demo:
    gr.Markdown("# Document Q&A Agent\nUpload a **public** PDF and ask questions. "
                "The agent grades its own retrieval, rewrites weak queries, "
                "and says 'I don't know' when evidence is missing.")
    f = gr.File(label="Upload a public PDF", file_types=[".pdf"])
    status = gr.Textbox(label="Status", interactive=False)
    f.upload(ingest, f, status)
    q = gr.Textbox(label="Your question")
    answer = gr.Markdown()
    sources = gr.Markdown()
    q.submit(ask, q, [answer, sources])

demo.launch(share=True, debug=True)

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://19e186559ec297e82d.gradio.live
